In [3]:
pip install selenium

  Using cached selenium-4.36.0-py3-none-any.whl (9.6 MB)
  Using cached websocket_client-1.9.0-py3-none-any.whl (82 kB)
     -------------------------------------- 131.6/131.6 KB 3.9 MB/s eta 0:00:00
     -------------------------------------- 152.9/152.9 KB 9.5 MB/s eta 0:00:00
  Using cached trio_websocket-0.12.2-py3-none-any.whl (21 kB)
  Using cached trio-0.31.0-py3-none-any.whl (512 kB)
  Using cached outcome-1.3.0.post0-py2.py3-none-any.whl (10 kB)
     ---------------------------------------- 71.0/71.0 KB 3.8 MB/s eta 0:00:00
  Using cached sniffio-1.3.1-py3-none-any.whl (10 kB)
  Using cached sortedcontainers-2.4.0-py2.py3-none-any.whl (29 kB)
  Using cached cffi-2.0.0-cp39-cp39-win_amd64.whl (182 kB)
  Using cached attrs-25.4.0-py3-none-any.whl (67 kB)
  Using cached wsproto-1.2.0-py3-none-any.whl (24 kB)
  Using cached PySocks-1.7.1-py3-none-any.whl (16 kB)
  Using cached pycparser-2.23-py3-none-any.whl (118 kB)
  Using cached h11-0.16.0-py3-none-any.whl (37 kB)
Note: you may

You should consider upgrading via the 'c:\Users\samue\Documents\datascientest-lol-draft_analyzer\env_csv\Scripts\python.exe -m pip install --upgrade pip' command.


In [5]:
pip install webdriver_manager


  Using cached webdriver_manager-4.0.2-py2.py3-none-any.whl (27 kB)
  Using cached requests-2.32.5-py3-none-any.whl (64 kB)
  Using cached python_dotenv-1.2.1-py3-none-any.whl (21 kB)
     -------------------------------------- 107.2/107.2 KB 6.5 MB/s eta 0:00:00


You should consider upgrading via the 'c:\Users\samue\Documents\datascientest-lol-draft_analyzer\env_csv\Scripts\python.exe -m pip install --upgrade pip' command.


In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.common.exceptions import WebDriverException
from webdriver_manager.chrome import ChromeDriverManager
import concurrent.futures
import time

DEBUG_PORT = 9222

def try_connect_existing_chrome():
    options = webdriver.ChromeOptions()
    options.add_argument("--start-maximized")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option(
        "debuggerAddress", f"127.0.0.1:{DEBUG_PORT}"
    )
    return webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)


def get_or_create_driver(timeout=5):
    start_time = time.time()
    while time.time() - start_time < timeout:
        try:
            print("🔁 Tentative de connexion à Chrome existant...")
            with concurrent.futures.ThreadPoolExecutor(max_workers=1) as executor:
                future = executor.submit(try_connect_existing_chrome)
                driver = future.result(timeout=timeout)
            print("✅ Connecté à Chrome existant")
            return driver
        except (WebDriverException, concurrent.futures.TimeoutError):
            print("⏳ Chrome non dispo ou timeout, retry...")
            time.sleep(0.5)

    # Après timeout → lancement d'un nouveau Chrome
    print("🚀 Timeout atteint → lancement d'un nouveau Chrome")
    options = webdriver.ChromeOptions()
    options.add_argument(f"--remote-debugging-port={DEBUG_PORT}")
    options.add_argument("--start-maximized")
    options.add_argument("--disable-blink-features=AutomationControlled")
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    print("🆕 Nouveau Chrome lancé avec debugging")
    return driver


In [2]:
def get_last_radix_buttons():
    """
    Récupère les boutons du dernier pop-up Radix ouvert.
    Utilise l'ID commençant par 'radix-' pour identifier le pop-up correct.
    """

    # 1️⃣ chercher tous les éléments dont l'ID commence par 'radix-'
    radix_roots = driver.find_elements(By.XPATH, "//*[starts-with(@id, 'radix-')]")
    if not radix_roots:

        return []

    # 2️⃣ prendre le dernier pop-up (le plus récemment ouvert)
    radix_root = radix_roots[-1]

    # 3️⃣ le div interne qui contient les boutons
    try:
        radix_div = radix_root.find_element(By.XPATH, "./div")
    except:

        return []

    # 4️⃣ récupérer les boutons
    buttons = radix_div.find_elements(By.TAG_NAME, "button")

    return buttons

In [ ]:
# import time
# from selenium.webdriver.common.by import By

# def init_parse(driver, scroll_pause=1.0, scroll_step=500):

#     champions_by_key = {}

#     scroll_top = 0
#     print("🚀 init_parse() démarré")

#     container_selector = (
#         "#root > main > div > div.flex.justify-center.gap-16 > "
#         "div.flex.flex-col.max-w-5xl.w-full.items-center.md\\:items-start.my-8.gap-8.px-12.lg\\:px-0 > "
#         "div.flex.flex-col.w-full.gap-12.bg-black-800.border.border-black-0\\/10.px-16.py-12.rounded-md > "
#         "div.flex.flex-col.gap-y-8.w-full.mb-48 > div:nth-child(3) > div"
#     )

#     while True:
#         driver.execute_script("window.scrollTo(0, arguments[0]);", scroll_top)
#         time.sleep(scroll_pause)

#         container = driver.find_element(By.CSS_SELECTOR, container_selector)

#         page_scroll_y = driver.execute_script("return window.pageYOffset;")
#         table_scroll_y = driver.execute_script(
#             "return arguments[0].scrollTop;", container
#         )

#         rows = container.find_elements(By.XPATH, "./div/div")

#         for idx, row in enumerate(rows):
#             text = row.text.strip()
#             if not text:
#                 continue

#             lines = text.splitlines()
#             label = lines[0]
#             name = lines[1] if len(lines) > 1 else "Unknown"

#             # ✅ récupération URL
#             url = None
#             try:
#                 link = row.find_element(By.XPATH, ".//a")
#                 url = link.get_attribute("href")
#             except:
#                 pass

#             if url:
#                 print(f"🔗 {label} → {url}")

#             key = text

#             if key not in champions_by_key:
#                 champions_by_key[key] = {
#                     "scroll_positions": [],
#                     "label": label,
#                     "numero": idx,
#                     "name": name,
#                     "url": url,   # ✅ stockée ici
#                 }

#             champions_by_key[key]["scroll_positions"].append({
#                 "page": page_scroll_y,
#                 "table": table_scroll_y
#             })

#         scroll_top += scroll_step
#         new_height = driver.execute_script("return document.body.scrollHeight")

#         if scroll_top >= new_height:
#             break

#     champions = []

#     for champ in champions_by_key.values():
#         positions = champ["scroll_positions"]

#         if len(positions) >= 2:
#             p1, p2 = positions[-2], positions[-1]
#             champ["scroll_page"] = (p1["page"] + p2["page"]) // 2
#             champ["scroll_table"] = (p1["table"] + p2["table"]) // 2
#         else:
#             champ["scroll_page"] = positions[-1]["page"]
#             champ["scroll_table"] = positions[-1]["table"]

#         del champ["scroll_positions"]
#         champions.append(champ)

#     print(f"✅ {len(champions)} champions collectés (scroll + url)")
#     return champions


In [3]:
import hashlib
from selenium.webdriver.common.by import By

ROLE_HASH_TO_TEXT = {
    "df14d23b35c9842bd8afa0db2b922aaf": "top",
    "bd9e54c883010f9bc2487e7d26a91b77": "jun",
    "3ce111209b6d69bee8498e94b02567ad": "mid",
    "6f1d8859e29002c2c45b527544e5a755": "adc",
    "3b66426a9b2beeca218cc726985f68b1": "sup",
}

def resolve_role_from_hash(svg_hash: str) -> str:
    role = ROLE_HASH_TO_TEXT.get(svg_hash)

    if role is None:
        print(f"⚠️ Hash de rôle inconnu : {svg_hash}")
        return "unknown"

    return role

# svg_hash = hashlib.md5(role_svg.encode("utf-8")).hexdigest()

def hash_svg_path(svg_element) -> str:
    """
    Extrait le path 'd' du SVG et retourne son hash MD5
    """
    paths = svg_element.find_elements(By.TAG_NAME, "path")
    if not paths:
        return None

    path_d = paths[0].get_attribute("d").strip()
    return hashlib.md5(path_d.encode("utf-8")).hexdigest()


In [4]:
driver = get_or_create_driver()
driver.get("https://dpm.lol/tierlist?tier=gold_plus")

🔁 Tentative de connexion à Chrome existant...
⏳ Chrome non dispo ou timeout, retry...
🚀 Timeout atteint → lancement d'un nouveau Chrome
🆕 Nouveau Chrome lancé avec debugging


In [5]:
# ------------------------------
# Imports
# ------------------------------
import time
import hashlib
from selenium.webdriver.common.by import By
from typing import Optional, Dict

# ------------------------------
# Mapping des rôles via hash SVG
# ------------------------------
ROLE_HASH_TO_TEXT_LINE = {
    "54d7bacd7686d25f9555c3381d5b3ecb": "jun",
    "d1a365179625b6191d515c69f5277dbd": "sup",
    "f84094b0fe98e4bf44fe62648e255e41": "adc",
    "6f7f06ca1bef87e71a35726cf843ad87": "top",
    "e4f796e42865301ea9dd362f979a2cdc": "mid",
}

def resolve_role_from_hash_line(svg_hash: str) -> str:
    role = ROLE_HASH_TO_TEXT_LINE.get(svg_hash)
    if role is None:
        print(f"⚠️ Hash de rôle inconnu : {svg_hash}")
        return "unknown"
    return role

# ------------------------------
# Parsing du texte complet du champion
# ------------------------------
def parse_champion_text(text: str) -> Optional[Dict]:
    """
    Attend un texte multi-lignes :
    1: label / numéro
    2: nom du champion
    3: % présence dans le rôle
    4: tier (S+, S, A...)
    5: winrate
    6: winrate+
    7: pickrate
    8: nombre de parties
    """
    try:
        lines = [l.strip() for l in text.split("\n") if l.strip()]
        if len(lines) < 8:
            # format court / inattendu
            return {
                "champion": lines[1] if len(lines) > 1 else "Unknown",
                "role_pickrate": lines[2] if len(lines) > 2 else "0%",
                "tier": lines[3] if len(lines) > 3 else "-",
                "winrate": lines[4] if len(lines) > 4 else "-",
                "winrate+": lines[5] if len(lines) > 5 else "-",
                "pickrate": lines[6] if len(lines) > 6 else "-",
                "games": lines[7] if len(lines) > 7 else "-",
            }

        # format normal
        return {
            "champion": lines[1],
            "role_pickrate": lines[2],
            "tier": lines[3],
            "winrate": lines[4],
            "winrate+": lines[5],
            "pickrate": lines[6],
            "games": lines[7],
        }

    except Exception as e:
        print("💥 ERREUR parse_champion_text")
        print("📄 Texte brut :", text)
        print("❌ Exception :", repr(e))
        return None

# ------------------------------
# Méthode principale : init_parse
# ------------------------------
def init_parse(driver, scroll_pause=1.0, scroll_step=500):
    """
    Scrape les champions avec scroll + toutes les stats.
    Retourne une liste de dictionnaires par champion.
    """
    champions_by_key = {}
    scroll_top = 0
    print("🚀 init_parse() démarré")

    container_selector = (
        "#root > main > div > div.flex.justify-center.gap-16 > "
        "div.flex.flex-col.max-w-5xl.w-full.items-center.md\\:items-start.my-8.gap-8.px-12.lg\\:px-0 > "
        "div.flex.flex-col.w-full.gap-12.bg-black-800.border.border-black-0\\/10.px-16.py-12.rounded-md > "
        "div.flex.flex-col.gap-y-8.w-full.mb-48 > div:nth-child(3) > div"
    )

    while True:
        driver.execute_script("window.scrollTo(0, arguments[0]);", scroll_top)
        time.sleep(scroll_pause)

        container = driver.find_element(By.CSS_SELECTOR, container_selector)

        page_scroll_y = driver.execute_script("return window.pageYOffset;")
        table_scroll_y = driver.execute_script("return arguments[0].scrollTop;", container)

        rows = container.find_elements(By.XPATH, "./div/div")

        for idx, row in enumerate(rows):
            text = row.text.strip()
            if not text:
                continue

            # récupération URL
            url = None
            try:
                link = row.find_element(By.XPATH, ".//a")
                url = link.get_attribute("href")
            except:
                pass

            # récupération SVG -> role
            svgs = row.find_elements(By.TAG_NAME, "svg")
            role = None
            if svgs:
                svg_html = svgs[0].get_attribute("outerHTML")
                svg_hash = hashlib.md5(svg_html.encode("utf-8")).hexdigest()
                role = resolve_role_from_hash_line(svg_hash)

            # parsing du texte complet du champion
            parsed = parse_champion_text(text)
            if parsed is None:
                continue

            key = text
            if key not in champions_by_key:
                champions_by_key[key] = {
                    "scroll_positions": [],
                    "label": text.splitlines()[0],
                    "numero": idx,
                    "name": text.splitlines()[1] if len(text.splitlines()) > 1 else "Unknown",
                    "url": url,
                    "role": role,
                    "role_pickrate": parsed["role_pickrate"],
                    "tier": parsed["tier"],
                    "winrate": parsed["winrate"],
                    "winrate_evol": parsed["winrate+"],
                    "pickrate": parsed["pickrate"],
                    "games": parsed["games"],
                }

            champions_by_key[key]["scroll_positions"].append({
                "page": page_scroll_y,
                "table": table_scroll_y
            })

        scroll_top += scroll_step
        new_height = driver.execute_script("return document.body.scrollHeight")
        if scroll_top >= new_height:
            break

    # Finalisation scroll positions
    champions = []
    for champ in champions_by_key.values():
        positions = champ["scroll_positions"]
        if len(positions) >= 2:
            p1, p2 = positions[-2], positions[-1]
            champ["scroll_page"] = (p1["page"] + p2["page"]) // 2
            champ["scroll_table"] = (p1["table"] + p2["table"]) // 2
        else:
            champ["scroll_page"] = positions[-1]["page"]
            champ["scroll_table"] = positions[-1]["table"]
        del champ["scroll_positions"]
        champions.append(champ)

    print(f"✅ {len(champions)} champions collectés (scroll + url + stats)")
    return champions


In [8]:
sauvegarde_champs_en_cours = {}

In [17]:
from selenium.webdriver.common.by import By
import time

import sys
sys.stdout.flush()

if __name__ == "__main__":

    param_lane = "top"
    param_synergy_or_matchup = "matchup"
    # param_lane = "jun"
    # param_synergy_or_matchup = "matchup"
    # param_lane = "mid"
    # param_synergy_or_matchup = "matchup"
    # param_lane = "adc"
    # param_synergy_or_matchup = "matchup"
    # param_lane = "sup"
    # param_synergy_or_matchup = "matchup"

    # param_lane = "top"
    # param_synergy_or_matchup = "synergy"
    # param_lane = "jun"
    # param_synergy_or_matchup = "synergy"
    # param_lane = "mid"
    # param_synergy_or_matchup = "synergy"
    # param_lane = "adc"
    # param_synergy_or_matchup = "synergy"
    # param_lane = "sup"
    # param_synergy_or_matchup = "synergy"
     
    # Variables pour suivre la combinaison active
    elo = None
    server = None
    patch = None

    big_champions_url_dict = {}
    big_champions_url_dict[f"{param_lane}_{param_synergy_or_matchup}"] = {}  # initialisation de la clé principale
    # sauvegarde_champs_en_cours[f"{param_lane}_{param_synergy_or_matchup}"] = {}
    root_key = f"{param_lane}_{param_synergy_or_matchup}"
    if root_key not in sauvegarde_champs_en_cours:
        sauvegarde_champs_en_cours[root_key] = {}

    filters_container_selector = (
        "#root > main > div > div.flex.justify-center.gap-16 > "
        "div.flex.flex-col.max-w-5xl.w-full.items-center.md\\:items-start.my-8.gap-8.px-12.lg\\:px-0 > "
        "div.flex.flex-col.w-full.gap-12.bg-black-800.border.border-black-0\\/10.px-16.py-12.rounded-md > "
        "div.flex.flex-col.lg\\:flex-row.items-center.justify-between.gap-16.lg\\:gap-24.w-full > "
        "div.flex.flex-row.items-center.justify-center.gap-8.lg\\:gap-16"
    )
    filters_container = driver.find_element(By.CSS_SELECTOR, filters_container_selector)

    print("✅ Containers trouvés")

    # =========================
    # 1️⃣ Ouvrir ELO et noter la liste des boutons
    # =========================
    filters_container.find_element(By.XPATH, ".//button[1]").click()
    time.sleep(0.5)
    elo_buttons = get_last_radix_buttons()
    print(f"\n🎯 ELO détectés : {[b.text for b in elo_buttons]}")

    # =========================
    # 2️⃣ Ouvrir SERVER et noter la liste des boutons
    # =========================
    filters_container.find_element(By.XPATH, ".//button[2]").click()
    time.sleep(0.5)
    server_buttons = get_last_radix_buttons()
    print(f"🌍 SERVER détectés : {[b.text for b in server_buttons]}")

    # =========================
    # 3️⃣ Ouvrir PATCH et noter la liste des boutons
    # =========================
    filters_container.find_element(By.XPATH, ".//div/button").click()
    time.sleep(0.5)
    patch_buttons = get_last_radix_buttons()
    print(f"🧩 PATCH détectés : {[b.text for b in patch_buttons]}")
    

    # =========================
    # BOUCLE SUR TOUTES LES COMBINAISONS
    # =========================


    # de elo_départ à elo_max
    # for i in range(numeloDepart | 0, max(AeloFin, len(elo_buttons))):
    # de elo_départ à elo_fin
    # for i in range(numeloDepart | 0, min(AelohFin, len(elo_buttons))): 
    # for i in range(0,max(1, len(elo_buttons))):
    for i in range(13,min(14, len(elo_buttons))):
    # for i in range(max(1, len(elo_buttons)) - 1, -1, -1):
    #     if not (i in (0, 1, 15)):
    #             continue  # 🔹 on skip les serveurs non désirés
        
        # 🔁 réouvrir la dropdown ELO
        filters_container.find_element(By.XPATH, ".//button[1]").click()
        time.sleep(0.7)
        elo_buttons = get_last_radix_buttons()
        elo_btn = elo_buttons[i]
        elo = elo_btn.text.strip()


        elo_btn.click()
        print(f"\n🎯 ELO [{i}] cliqué → {elo}")

        big_champions_url_dict[f"{param_lane}_{param_synergy_or_matchup}"][f"{elo}"] = {}
        # sauvegarde_champs_en_cours[f"{param_lane}_{param_synergy_or_matchup}"][f"{elo}"] = {}
        sauvegarde_champs_en_cours.setdefault(root_key, {}).setdefault(elo, {})
        
        time.sleep(1)
        print("1")
        time.sleep(1)
        print("2")

        # de server_départ à server_max
        # for j in range(numServerDepart | 0, max(AServerFin, len(server_buttons))):
        # de server_départ à server_fin
        # for j in range(numServerDepart | 0, min(AServerFin, len(server_buttons))): 
        # for j in range(7, min(8, len(server_buttons))):
        #     if ( ((j >= 3) and (j <= 10) and (j != 4) and(j!=7)) ): #7 pour LAS pour les tests


        for j in range(max(8, len(server_buttons)) - 1, -1, -1):
            if ((j >= 2) and (j <= 10)):
                continue  # 🔹 on skip les serveurs non désirés
            
            # 🔁 réouvrir la dropdown SERVER
            filters_container.find_element(By.XPATH, ".//button[2]").click()
            time.sleep(0.7)
            server_buttons = get_last_radix_buttons()
            server_btn = server_buttons[j]
            server = server_btn.text.strip()

            big_champions_url_dict[f"{param_lane}_{param_synergy_or_matchup}"][f"{elo}"][f"{server}"] = {}
            # sauvegarde_champs_en_cours[f"{param_lane}_{param_synergy_or_matchup}"][f"{elo}"][f"{server}"] = {}
            sauvegarde_champs_en_cours.setdefault(root_key, {}).setdefault(elo, {}).setdefault(server, {})

            server_btn.click()
            print(f"  🌍 SERVER [{j}] cliqué → {server}")
            time.sleep(1)
            print("1")
            time.sleep(1)
            print("2")
            time.sleep(1)
            print("3")
            filters_container = driver.find_element(By.CSS_SELECTOR, filters_container_selector)


            # de patch_départ à patch_max
            # for k in range(numPatchDepart | 0, max(APatchFin, len(patch_buttons))):
            # de patch_départ à patch_fin
            # for k in range(numPatchDepart | 0, min(APatchFin, len(patch_buttons))): 
            # for k in range(5, min(6, len(patch_buttons))):  # 🔹 on limite à 3 itérations pour tester          
            for k in range(3, max(5, len(patch_buttons))):  # 🔹 on limite à 3 itérations pour tester   
                if ((k ==0 ) or (k == 1) or (k==2) or (k == 4) or (k == 5) or (k == 7) or (k == 8) ):
                    continue  # 🔹 on skip les serveurs non désirés       
                # 🔁 réouvrir la dropdown PATCH
                filters_container.find_element(By.XPATH, ".//div/button").click()
                time.sleep(1)

                # 🔹 récupérer à nouveau les boutons PATCH pour éviter StaleElementReference
                patch_buttons = get_last_radix_buttons()
                time.sleep(1)
                print("click sur les patchs")
                print(f"🧩 PATCH mis à jour : {[b.text for b in patch_buttons]}")
                patch_btn = patch_buttons[k]

                # 🔹 cliquer sur le kème bouton
                patch = patch_btn.text.strip()
                patch_btn.click()
                time.sleep(0.7)

                print(f"    🧩 PATCH [{k}] cliqué → {patch}")

                # ✅ COMBINAISON ACTIVE
                print(f"    ✅ COMBINAISON ACTIVE : ELO={elo}, SERVER={server}, PATCH={patch}")
                time.sleep(1)
                print("1")
                time.sleep(1)
                print("2")
                time.sleep(1)
                print("3")
                
                filters_container = driver.find_element(By.CSS_SELECTOR, filters_container_selector)
                

                champions = init_parse(driver)
                big_champions_url_dict[f"{param_lane}_{param_synergy_or_matchup}"][f"{elo}"][f"{server}"][f"{patch}"] = champions
                # sauvegarde_champs_en_cours[f"{param_lane}_{param_synergy_or_matchup}"][f"{elo}"][f"{server}"][f"{patch}"] = champions
                sauvegarde_champs_en_cours.setdefault(root_key, {}).setdefault(elo, {}).setdefault(server, {}).setdefault(patch, champions)
                
                driver.execute_script("window.scrollTo(0, arguments[0]);", 0)




✅ Containers trouvés

🎯 ELO détectés : ['Challenger', 'Grandmaster', 'Master+', 'Master', 'Diamond+', 'Diamond', 'Emerald+', 'Emerald', 'Platinum+', 'Platinum', 'Gold+', 'Gold', 'Silver+', 'Bronze', 'Iron', 'TOUT']
🌍 SERVER détectés : ['EUW', 'KR', 'NA', 'BR', 'EUNE', 'JP', 'LAN', 'LAS', 'OCE', 'RU', 'TR', 'VN', 'TOUT']
🧩 PATCH détectés : ['7days', '14days', '30days', '16.3', '16.2', '16.1', '15.24', '15.23', '15.22']

🎯 ELO [13] cliqué → Bronze
1
2
  🌍 SERVER [12] cliqué → TOUT
1
2
3
click sur les patchs
🧩 PATCH mis à jour : ['7days', '14days', '30days', '16.3', '16.2', '16.1', '15.24', '15.23', '15.22']
    🧩 PATCH [3] cliqué → 16.3
    ✅ COMBINAISON ACTIVE : ELO=Bronze, SERVER=TOUT, PATCH=16.3
1
2
3
🚀 init_parse() démarré
✅ 209 champions collectés (scroll + url + stats)
click sur les patchs
🧩 PATCH mis à jour : ['7days', '14days', '30days', '16.3', '16.2', '16.1', '15.24', '15.23', '15.22']
    🧩 PATCH [6] cliqué → 15.24
    ✅ COMBINAISON ACTIVE : ELO=Bronze, SERVER=TOUT, PATCH=15.2

In [6]:
def build_champion_dict(obj):
    """
    Transforme le JSON imbriqué en :
    {
        "lane_mode_elo_server_patch": [champions...],
        ...
    }
    """

    result = {}

    def walk(node, path_keys):
        # cas feuille = liste de champions
        if isinstance(node, list):
            if len(path_keys) == 4:
                lane_mode, elo, server, patch = path_keys
                key = f"{lane_mode}_{elo}_{server}_{patch}"
                result[key] = node
                print(f"✅ Liste champions trouvée → {key} ({len(node)} champions)")
            return

        if isinstance(node, dict):
            for k, v in node.items():
                walk(v, path_keys + [k])

    walk(obj, [])
    return result


In [19]:
champion_dict = build_champion_dict(sauvegarde_champs_en_cours)

print(len(champion_dict))
print(list(champion_dict.keys())[:5])
for k, v in champion_dict.items():
    print(k, len(v))

✅ Liste champions trouvée → top_matchup_Bronze_TOUT_16.3 (209 champions)
✅ Liste champions trouvée → top_matchup_Bronze_TOUT_15.24 (214 champions)
✅ Liste champions trouvée → top_matchup_Bronze_VN_16.3 (200 champions)
✅ Liste champions trouvée → top_matchup_Bronze_VN_15.24 (200 champions)
✅ Liste champions trouvée → top_matchup_Bronze_KR_16.3 (195 champions)
✅ Liste champions trouvée → top_matchup_Bronze_KR_15.24 (200 champions)
✅ Liste champions trouvée → top_matchup_Bronze_EUW_16.3 (205 champions)
✅ Liste champions trouvée → top_matchup_Bronze_EUW_15.24 (214 champions)
8
['top_matchup_Bronze_TOUT_16.3', 'top_matchup_Bronze_TOUT_15.24', 'top_matchup_Bronze_VN_16.3', 'top_matchup_Bronze_VN_15.24', 'top_matchup_Bronze_KR_16.3']
top_matchup_Bronze_TOUT_16.3 209
top_matchup_Bronze_TOUT_15.24 214
top_matchup_Bronze_VN_16.3 200
top_matchup_Bronze_VN_15.24 200
top_matchup_Bronze_KR_16.3 195
top_matchup_Bronze_KR_15.24 200
top_matchup_Bronze_EUW_16.3 205
top_matchup_Bronze_EUW_15.24 214


In [7]:
import csv
import json

def save_champion_dict_to_csv(champion_dict, csv_path):
    """
    Enregistre champion_dict dans un CSV.
    Format :
        key ; json_list_of_champions
    """

    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["key", "champions_json"])

        for key, champions in champion_dict.items():
            champions_json = json.dumps(champions, ensure_ascii=False)
            writer.writerow([key, champions_json])

    print(f"✅ champion_dict sauvegardé → {csv_path}")

def load_champion_dict_from_csv(csv_path):
    """
    Recharge le dictionnaire depuis le CSV produit ci-dessus.
    """

    champion_dict = {}

    with open(csv_path, "r", newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)

        for row in reader:
            key = row["key"]
            champions = json.loads(row["champions_json"])
            champion_dict[key] = champions

    print(f"✅ champion_dict chargé ← {csv_path}")
    return champion_dict



In [ ]:
save_champion_dict_to_csv(champion_dict, "aaa.csv")

# champion_dict_bis = load_champion_dict_from_csv("champions_simple_WR_bronz_dump.csv")

✅ champion_dict sauvegardé → champions_simple_WR_bronz_dump.csv


In [8]:
# save_champion_dict_to_csv(champion_dict, "aaa.csv")

champion_dict_bis = load_champion_dict_from_csv("champions_simple_WR_compressed_dump.csv")

✅ champion_dict chargé ← champions_simple_WR_compressed_dump.csv


In [13]:
driver.get("https://dpm.lol/tierlist?tier=gold_plus")

In [12]:


options = webdriver.ChromeOptions()
options.add_argument(f"--remote-debugging-port={DEBUG_PORT}")
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

In [19]:
driver = get_or_create_driver()
driver.get("https://dpm.lol/tierlist?tier=gold_plus")

🔁 Tentative de connexion à Chrome existant...
⏳ Chrome non dispo ou timeout, retry...
🚀 Timeout atteint → lancement d'un nouveau Chrome
🆕 Nouveau Chrome lancé avec debugging


In [13]:
#########################################
# on va créer le DF de tous les WR simples
champion_dict_compressed = load_champion_dict_from_csv("champions_simple_WR_compressed_dump.csv")
champion_dict_bronz = load_champion_dict_from_csv("champions_simple_WR_bronz_dump.csv")
champion_dict_complet = load_champion_dict_from_csv("champions_dump_complet.csv")
champion_dict_minimal = load_champion_dict_from_csv("champions_simple_WR_compressed_minimal_dump.csv")


✅ champion_dict chargé ← champions_simple_WR_compressed_dump.csv
✅ champion_dict chargé ← champions_simple_WR_bronz_dump.csv
✅ champion_dict chargé ← champions_dump_complet.csv
✅ champion_dict chargé ← champions_simple_WR_compressed_minimal_dump.csv


In [15]:
# champion_dict = build_champion_dict(sauvegarde_champs_en_cours)

# print(len(champion_dict_compressed))
# print(list(champion_dict_compressed.keys())[:5])
# for k, v in champion_dict_compressed.items():
#     print(k, len(v))

# print(len(champion_dict_bronz))
# print(list(champion_dict_bronz.keys())[:5])
# for k, v in champion_dict_bronz.items():
#     print(k, len(v))

# print(len(champion_dict_complet))
# print(list(champion_dict_complet.keys())[:5])
# for k, v in champion_dict_complet.items():
#     print(k, len(v))

print(len(champion_dict_minimal))
print(list(champion_dict_minimal.keys())[:5])
for k, v in champion_dict_minimal.items():
    print(k, len(v))

21
['top_matchup_TOUT_TOUT_16.3', 'top_matchup_TOUT_TOUT_15.24', 'top_matchup_TOUT_NA_16.3', 'top_matchup_TOUT_NA_15.24', 'top_matchup_TOUT_KR_16.3']
top_matchup_TOUT_TOUT_16.3 212
top_matchup_TOUT_TOUT_15.24 219
top_matchup_TOUT_NA_16.3 220
top_matchup_TOUT_NA_15.24 224
top_matchup_TOUT_KR_16.3 201
top_matchup_TOUT_KR_15.24 203
top_matchup_TOUT_EUW_16.3 215
top_matchup_TOUT_EUW_15.24 221
top_matchup_Bronze_TOUT_16.3 209
top_matchup_Grandmaster_TOUT_16.3 208
top_matchup_Grandmaster_TOUT_15.24 219
top_matchup_Grandmaster_KR_16.3 143
top_matchup_Grandmaster_KR_15.24 196
top_matchup_Grandmaster_EUW_16.3 178
top_matchup_Grandmaster_EUW_15.24 215
top_matchup_Challenger_TOUT_16.3 210
top_matchup_Challenger_TOUT_15.24 219
top_matchup_Challenger_KR_16.3 77
top_matchup_Challenger_KR_15.24 135
top_matchup_Challenger_EUW_16.3 98
top_matchup_Challenger_EUW_15.24 123


In [ ]:
print(champion_dict_minimal["top_matchup_TOUT_TOUT_15.24"])

[{'label': '1', 'numero': 0, 'name': 'Miss Fortune', 'url': 'https://dpm.lol/champions/MissFortune/build?lane=bottom&tier=all&platform=all&timeframe=15.24', 'role': 'adc', 'role_pickrate': '97.3%', 'tier': 'S+', 'winrate': '52.2%', 'winrate_evol': '-0.0%', 'pickrate': '22.6%', 'games': '5\u202f439\u202f399', 'scroll_page': 250, 'scroll_table': 0}, {'label': '2', 'numero': 1, 'name': 'Briar', 'url': 'https://dpm.lol/champions/Briar/build?lane=jungle&tier=all&platform=all&timeframe=15.24', 'role': 'jun', 'role_pickrate': '92.1%', 'tier': 'S+', 'winrate': '53.2%', 'winrate_evol': '+1.5%', 'pickrate': '5.9%', 'games': '1\u202f424\u202f721', 'scroll_page': 250, 'scroll_table': 0}, {'label': '3', 'numero': 2, 'name': 'Malphite', 'url': 'https://dpm.lol/champions/Malphite/build?lane=top&tier=all&platform=all&timeframe=15.24', 'role': 'top', 'role_pickrate': '47.4%', 'tier': 'S+', 'winrate': '52.5%', 'winrate_evol': '+0.1%', 'pickrate': '8.1%', 'games': '1\u202f955\u202f012', 'scroll_page': 25

In [18]:
import pandas as pd

dfs = []

for key, value in champion_dict_minimal.items():
    
    # split de la clé
    parts = key.split("_")
    
    # récupération des infos
    elo = parts[2]
    server = parts[3]
    patch = parts[4]
    
    # création du DataFrame
    temp_df = pd.DataFrame(value)
    
    # ajout des colonnes
    temp_df["elo"] = elo
    temp_df["server"] = server
    temp_df["patch"] = patch
    
    dfs.append(temp_df)

# concat
df = pd.concat(dfs, ignore_index=True)

# 🔥 placer les colonnes au début
cols = ["elo", "server", "patch"] + [c for c in df.columns if c not in ["elo", "server", "patch"]]
df = df[cols]

df.head(230)

,elo,server,patch,label,numero,name,url,role,role_pickrate,tier,winrate,winrate_evol,pickrate,games,scroll_page,scroll_table
0,TOUT,TOUT,16.3,1,0,Briar,https://dpm.lol/champions/Briar/build?lane=jun...,jun,91.1%,S+,52.8%,+1.5%,6.7%,599 576,250.0,0
1,TOUT,TOUT,16.3,2,1,Miss Fortune,https://dpm.lol/champions/MissFortune/build?la...,adc,97.0%,S+,51.7%,-0.0%,15.5%,1 386 160,250.0,0
2,TOUT,TOUT,16.3,3,2,Malzahar,https://dpm.lol/champions/Malzahar/build?lane=...,mid,92.2%,S+,52.3%,+0.0%,8.1%,724 180,250.0,0
3,TOUT,TOUT,16.3,4,3,Swain,https://dpm.lol/champions/Swain/build?lane=bot...,adc,18.6%,S+,54.6%,-0.1%,1.7%,154 988,250.0,0
4,TOUT,TOUT,16.3,5,4,Jinx,https://dpm.lol/champions/Jinx/build?lane=bott...,adc,99.5%,S+,51.6%,-0.2%,16.6%,1 485 175,500.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
225,TOUT,TOUT,15.24,14,13,Warwick,https://dpm.lol/champions/Warwick/build?lane=j...,jun,82.0%,S,51.9%,+0.1%,6.6%,1 585 624,750.0,0
226,TOUT,TOUT,15.24,15,14,Smolder,https://dpm.lol/champions/Smolder/build?lane=b...,adc,82.4%,S,51.5%,-0.1%,9.6%,2 310 043,750.0,0
227,TOUT,TOUT,15.24,16,15,Malphite,https://dpm.lol/champions/Malphite/build?lane=...,jun,35.4%,S,51.5%,-0.2%,6.1%,1 457 774,750.0,0
228,TOUT,TOUT,15.24,17,16,Morgana,https://dpm.lol/champions/Morgana/build?lane=u...,sup,74.9%,S,50.9%,+0.0%,8.1%,1 940 416,750.0,0


In [19]:
import pandas as pd

df = pd.concat(
    [
        pd.DataFrame(v).assign(
            elo=k.split("_")[2],
            server=k.split("_")[3],
            patch=k.split("_")[4]
        )
        for k, v in champion_dict_minimal.items()
    ],
    ignore_index=True
)

# ✅ placer elo / server / patch au début
df = df[["elo", "server", "patch"] + [c for c in df.columns if c not in ["elo", "server", "patch"]]]

# ✅ supprimer colonnes inutiles (sans erreur si absentes)
df = df.drop(columns=["numero", "scroll_page", "scroll_table", "url"], errors="ignore")

df.head()

,elo,server,patch,label,name,role,role_pickrate,tier,winrate,winrate_evol,pickrate,games
0,TOUT,TOUT,16.3,1,Briar,jun,91.1%,S+,52.8%,+1.5%,6.7%,599 576
1,TOUT,TOUT,16.3,2,Miss Fortune,adc,97.0%,S+,51.7%,-0.0%,15.5%,1 386 160
2,TOUT,TOUT,16.3,3,Malzahar,mid,92.2%,S+,52.3%,+0.0%,8.1%,724 180
3,TOUT,TOUT,16.3,4,Swain,adc,18.6%,S+,54.6%,-0.1%,1.7%,154 988
4,TOUT,TOUT,16.3,5,Jinx,adc,99.5%,S+,51.6%,-0.2%,16.6%,1 485 175


In [20]:
df.shape

(3945, 12)

In [22]:
for col in df.columns:
    print("\n", col)
    print(df[col].unique()[:10])  # limite à 10 pour éviter les gros prints


 elo
['TOUT' 'Bronze' 'Grandmaster' 'Challenger']

 server
['TOUT' 'NA' 'KR' 'EUW']

 patch
['16.3' '15.24']

 label
['1' '2' '3' '4' '5' '6' '7' '8' '9' '10']

 name
['Briar' 'Miss Fortune' 'Malzahar' 'Swain' 'Jinx' 'Nami' 'Braum'
 'Dr.Mundo' 'Mordekaiser' 'Kayle']

 role
['jun' 'adc' 'mid' 'sup' 'top']

 role_pickrate
['91.1%' '97.0%' '92.2%' '18.6%' '99.5%' '99.9%' '99.7%' '45.1%' '88.8%'
 '85.8%']

 tier
['S+' 'S' 'A' 'B' 'C' 'D']

 winrate
['52.8%' '51.7%' '52.3%' '54.6%' '51.6%' '52.1%' '51.9%' '52.9%' '51.1%'
 '52.5%']

 winrate_evol
['+1.5%' '-0.0%' '+0.0%' '-0.1%' '-0.2%' '-0.9%' '+0.1%' '+0.4%' '+1.1%'
 '-0.3%']

 pickrate
['6.7%' '15.5%' '8.1%' '1.7%' '16.6%' '11.8%' '8.0%' '4.5%' '5.6%' '12.0%']

 games
['599\u202f576' '1\u202f386\u202f160' '724\u202f180' '154\u202f988'
 '1\u202f485\u202f175' '1\u202f056\u202f128' '713\u202f525' '399\u202f990'
 '726\u202f440' '502\u202f463']


In [23]:
import pandas as pd

dfs_bronz = []

for key, value in champion_dict_bronz.items():
    
    # split de la clé
    parts = key.split("_")
    
    # récupération des infos
    elo = parts[2]
    server = parts[3]
    patch = parts[4]
    
    # création du DataFrame
    temp_df = pd.DataFrame(value)
    
    # ajout des colonnes
    temp_df["elo"] = elo
    temp_df["server"] = server
    temp_df["patch"] = patch
    
    dfs_bronz.append(temp_df)

# concat
df_bronz = pd.concat(dfs_bronz, ignore_index=True)

# 🔥 placer les colonnes au début
cols = ["elo", "server", "patch"] + [c for c in df_bronz.columns if c not in ["elo", "server", "patch"]]
df_bronz = df_bronz[cols]

df_bronz.head(230)

,elo,server,patch,label,numero,name,url,role,role_pickrate,tier,winrate,winrate_evol,pickrate,games,scroll_page,scroll_table
0,Bronze,TOUT,16.3,1,0,Briar,https://dpm.lol/champions/Briar/build?lane=jun...,jun,92.3%,S+,54.2%,+1.8%,8.1%,65 117,250.0,0
1,Bronze,TOUT,16.3,2,1,Malzahar,https://dpm.lol/champions/Malzahar/build?lane=...,mid,90.6%,S+,53.8%,+0.1%,8.3%,66 264,250.0,0
2,Bronze,TOUT,16.3,3,2,Miss Fortune,https://dpm.lol/champions/MissFortune/build?la...,adc,96.8%,S+,52.6%,+0.1%,26.9%,214 855,250.0,0
3,Bronze,TOUT,16.3,4,3,Dr.Mundo,https://dpm.lol/champions/DrMundo/build?lane=j...,jun,42.6%,S+,54.6%,+0.4%,4.9%,39 537,250.0,0
4,Bronze,TOUT,16.3,5,4,Yorick,https://dpm.lol/champions/Yorick/build?lane=to...,top,90.1%,S+,54.0%,-0.2%,6.7%,53 859,500.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
225,Bronze,TOUT,15.24,17,16,Swain,https://dpm.lol/champions/Swain/build?lane=mid...,mid,18.4%,S+,54.3%,-0.9%,1.9%,52 324,750.0,0
226,Bronze,TOUT,15.24,18,17,Lux,https://dpm.lol/champions/Lux/build?lane=utili...,sup,72.1%,S+,50.7%,-0.0%,17.6%,471 588,750.0,0
227,Bronze,TOUT,15.24,19,18,Malphite,https://dpm.lol/champions/Malphite/build?lane=...,jun,33.5%,S+,52.4%,-0.4%,6.2%,166 804,750.0,0
228,Bronze,TOUT,15.24,20,19,Viego,https://dpm.lol/champions/Viego/build?lane=jun...,jun,97.5%,S+,51.1%,+0.0%,11.7%,312 726,750.0,0


In [24]:
import pandas as pd

df_bronz = pd.concat(
    [
        pd.DataFrame(v).assign(
            elo=k.split("_")[2],
            server=k.split("_")[3],
            patch=k.split("_")[4]
        )
        for k, v in champion_dict_bronz.items()
    ],
    ignore_index=True
)

# ✅ placer elo / server / patch au début
df_bronz = df_bronz[["elo", "server", "patch"] + [c for c in df_bronz.columns if c not in ["elo", "server", "patch"]]]

# ✅ supprimer colonnes inutiles (sans erreur si absentes)
df_bronz = df_bronz.drop(columns=["numero", "scroll_page", "scroll_table", "url"], errors="ignore")

df_bronz.head()

,elo,server,patch,label,name,role,role_pickrate,tier,winrate,winrate_evol,pickrate,games
0,Bronze,TOUT,16.3,1,Briar,jun,92.3%,S+,54.2%,+1.8%,8.1%,65 117
1,Bronze,TOUT,16.3,2,Malzahar,mid,90.6%,S+,53.8%,+0.1%,8.3%,66 264
2,Bronze,TOUT,16.3,3,Miss Fortune,adc,96.8%,S+,52.6%,+0.1%,26.9%,214 855
3,Bronze,TOUT,16.3,4,Dr.Mundo,jun,42.6%,S+,54.6%,+0.4%,4.9%,39 537
4,Bronze,TOUT,16.3,5,Yorick,top,90.1%,S+,54.0%,-0.2%,6.7%,53 859


In [25]:
df_bronz.shape

(1637, 12)

In [26]:
import pandas as pd

# concat
df_all = pd.concat([df, df_bronz], ignore_index=True)
print(df_all.shape)

# suppression doublons (toutes colonnes)
df_all = df_all.drop_duplicates()

# reset index propre
df_all = df_all.reset_index(drop=True)

print(df_all.shape)
df_all.head()

(5582, 12)
(5373, 12)


,elo,server,patch,label,name,role,role_pickrate,tier,winrate,winrate_evol,pickrate,games
0,TOUT,TOUT,16.3,1,Briar,jun,91.1%,S+,52.8%,+1.5%,6.7%,599 576
1,TOUT,TOUT,16.3,2,Miss Fortune,adc,97.0%,S+,51.7%,-0.0%,15.5%,1 386 160
2,TOUT,TOUT,16.3,3,Malzahar,mid,92.2%,S+,52.3%,+0.0%,8.1%,724 180
3,TOUT,TOUT,16.3,4,Swain,adc,18.6%,S+,54.6%,-0.1%,1.7%,154 988
4,TOUT,TOUT,16.3,5,Jinx,adc,99.5%,S+,51.6%,-0.2%,16.6%,1 485 175


In [ ]:
df_all.to_csv("df_Simple_WR_FULL.csv", index=False, encoding="utf-8")

: 